In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

def normalize_columns(df):
    """Standardizes column names."""
    for c in df.columns:
        df = df.withColumnRenamed(c, c.strip().lower().replace(" ", "_"))
    return df

def cast_timestamp(df, col):
    return df.withColumn(col, F.to_timestamp(F.col(col)))

def cast_float(df, col):
    return df.withColumn(col, F.col(col).cast("double"))

def cast_int(df, col):
    return df.withColumn(col, F.col(col).cast("int"))

In [0]:
df_store = spark.table("workspace.bronze.store")
df_store = normalize_columns(df_store)

df_store = (
    df_store
    .drop("_file", "_line", "_modified", "_fivetran_synced")
    .withColumn("store_id", F.col("store_id").cast("int"))
    .withColumn("opened_year", F.col("opened_year").cast("int"))
    .dropDuplicates()
)

# Backfill null manager_name using manager_id
manager_lookup = (
    df_store.filter(F.col("manager_name").isNotNull())
    .select("manager_id", F.col("manager_name").alias("manager_name_lookup")).distinct()
)
df_store = (
    df_store
    .join(manager_lookup, on="manager_id", how="left")
    .withColumn("manager_name", F.coalesce(F.col("manager_name"), F.col("manager_name_lookup")))
    .drop("manager_name_lookup")
)

df_store.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("workspace.silver.store")

In [0]:
df_order = spark.table("workspace.bronze.order")
    df_order = normalize_columns(df_order)

df_order = (
    df_order
    .drop("_file", "_line", "_modified", "_fivetran_synced")
    .withColumn("store_id", F.col("store_id").cast("int"))
    .dropDuplicates()
)

df_order.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("workspace.silver.order")

In [0]:
df_survey = spark.table("workspace.bronze.customer_survey")
df_survey = normalize_columns(df_survey)

df_survey = (
    df_survey
    .drop("_file", "_line", "_modified", "_fivetran_synced")
    .withColumn("delivered_on_time_rating", F.col("delivered_on_time_rating").cast("int"))
    .withColumn("work_quality_rating", F.col("work_quality_rating").cast("int"))
    .withColumn("cleanliness_rating", F.col("cleanliness_rating").cast("int"))
    .withColumn("communication_rating", F.col("communication_rating").cast("int"))
    .withColumn("overall_satisfaction_rating", F.col("overall_satisfaction_rating").cast("int"))
    .dropDuplicates()
)

# Fix responded_flag nulls: true if response_date exists, false otherwise
df_survey = df_survey.withColumn(
    "responded_flag",
    F.when(F.col("responded_flag").isNull() & F.col("survey_response_date").isNotNull(), F.lit(True))
    .when(F.col("responded_flag").isNull() & F.col("survey_response_date").isNull(), F.lit(False))
    .otherwise(F.col("responded_flag"))
)

rating_cols = [
    "delivered_on_time_rating", "work_quality_rating",
    "cleanliness_rating", "communication_rating", "overall_satisfaction_rating"
]

# Join with order table to get store_id
df_order = spark.table("workspace.silver.order").select("order_id", "store_id")
df_survey = df_survey.join(df_order, on="order_id", how="left")

# Compute store-level average ratings (from responded surveys only)
store_avg = (
    df_survey.filter(F.col("responded_flag") == True)
    .groupBy("store_id")
    .agg(*[F.round(F.avg(c)).cast("int").alias(f"{c}_avg") for c in rating_cols])
)

# Compute global averages as fallback
global_avg = (
    df_survey.filter(F.col("responded_flag") == True)
    .agg(*[F.round(F.avg(c)).cast("int").alias(f"{c}_global") for c in rating_cols])
)

# Join store averages and fill nulls
df_survey = df_survey.join(store_avg, on="store_id", how="left")
df_survey = df_survey.crossJoin(global_avg)

for c in rating_cols:
    df_survey = df_survey.withColumn(
        c,
        F.coalesce(F.col(c), F.col(f"{c}_avg"), F.col(f"{c}_global"))
    )

# Drop helper columns and store_id (not in original schema)
cols_to_drop = [f"{c}_avg" for c in rating_cols] + [f"{c}_global" for c in rating_cols] + ["store_id"]
df_survey = df_survey.drop(*cols_to_drop)

df_survey.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("workspace.silver.customer_survey")

In [0]:
df_estimate = spark.table("workspace.bronze.estimate")
df_estimate = normalize_columns(df_estimate)

df_estimate = (
    df_estimate
    .drop("_file", "_line", "_modified", "_fivetran_synced")
    .withColumn("version_no", F.col("version_no").cast("int"))
    .dropDuplicates()
)

df_estimate.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("workspace.silver.estimate")

In [0]:
df_invoice = spark.table("workspace.bronze.invoice")
df_invoice = normalize_columns(df_invoice)

df_invoice = (
    df_invoice
    .drop("_file", "_line", "_modified", "_fivetran_synced")
    .withColumn("invoice_amount", F.col("invoice_amount").cast("double"))
    .dropDuplicates()
)

df_invoice.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("workspace.silver.invoice")

In [0]:
df_budget = spark.table("workspace.bronze.ns_budget")
df_budget = normalize_columns(df_budget)

df_budget = (
    df_budget
    .drop("_file", "_line", "_modified", "_fivetran_synced")
    .withColumn("ns_store_id", F.col("ns_store_id").cast("int"))
    .withColumn("budget_amount", F.col("budget_amount").cast("double"))
    .dropDuplicates()
)

df_budget.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("workspace.silver.ns_budget")